# FINAL TEST — test-2025, run ONCE

**The rules:**

1. Run this notebook **once**, after `evaluate.ipynb` and (optionally) `train_gru.ipynb` are done and the winner is decided on **val**
2. Thresholds come from `eval_summary.json` (chosen on val) — **nothing** is tuned here
3. Whatever numbers come out go in the report. No re-runs after seeing them, no "just one tweak". If you change the model afterwards, 2025 is burned as a test set and you must say so in the report.

Outputs: `final_report.json`, `final_station_breakdown.csv` — the numbers for the BMA report and evaluator point #12.

In [1]:
import json
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.metrics import average_precision_score

pd.set_option("display.width", 160, "display.max_columns", 30)

## Configuration — set the winner, chosen on val

In [2]:
WINNER = "lgbm"          # "lgbm" or "gru" — per evaluate/train_gru verdicts

TRAINING_DIR = Path("../data/training")
ARTIFACTS = Path("../models/artifacts")
HORIZONS = [1, 3, 6]
TIERS = [5, 15, 30]
QUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]

meta = json.loads((TRAINING_DIR / "features.json").read_text())
FEATURES = meta["features"] + ["station_code"]
ev = json.loads((ARTIFACTS / "eval_summary.json").read_text())
THRESHOLDS = ev["thresholds"]          # frozen on val
VALID_MIN = ev["config"]["VALID_MIN"]
calibrators = joblib.load(ARTIFACTS / "calibrators.joblib")
print("winner:", WINNER, "| thresholds:",
      {k: round(v, 4) for k, v in list(THRESHOLDS.items())[:3]}, "...")

winner: lgbm | thresholds: {'ge5_1h': 0.8624, 'ge15_1h': 0.9186, 'ge30_1h': 1.0} ...


## Load test-2025

In [3]:
label_cols = ([f"y_maxdepth_{h}h" for h in HORIZONS]
              + [f"y_valid_{h}h" for h in HORIZONS]
              + [f"y_ge{t}_{h}h" for h in HORIZONS for t in TIERS])
cols = ["station_code", "site_timestamp"] + meta["features"] + label_cols
table = pq.read_table(TRAINING_DIR / "test.parquet", columns=cols)
schema = pa.schema([pa.field(f.name, pa.float32())
                    if f.type == pa.float64() else f for f in table.schema])
test = table.cast(schema).to_pandas(self_destruct=True)
test["station_code"] = test["station_code"].astype("category")
print(f"test: {len(test):,} rows, {test.station_code.nunique()} stations, "
      f"{test.site_timestamp.min().date()} -> {test.site_timestamp.max().date()}")

test: 3,348,382 rows, 100 stations, 2025-01-01 -> 2025-12-31


## Score the winner on test

In [4]:
def score_lgbm(df):
    out = {}
    for h in HORIZONS:
        m = (df[f"y_valid_{h}h"] >= VALID_MIN).to_numpy()
        for t in TIERS:
            b = lgb.Booster(model_file=str(ARTIFACTS / f"clf_ge{t}_{h}h.txt"))
            out[(t, h)] = (m, b.predict(df.loc[m, FEATURES]))
    return out

if WINNER == "lgbm":
    scores = score_lgbm(test)
else:
    raise NotImplementedError(
        "WINNER='gru': run the GRU inference cell at the bottom of this "
        "notebook first (it defines score_gru), then set scores = score_gru(test).")
print("scored", len(scores), "targets")

scored 9 targets


## Final metrics (hybrid rule, val thresholds) — full year and monsoon (May–Oct)

In [5]:
def f2_at(y, pred):
    tp = int((pred & (y == 1)).sum())
    if tp == 0:
        return 0.0, 0.0, 0.0
    p, r = tp / pred.sum(), tp / (y == 1).sum()
    return 5 * p * r / (4 * p + r), p, r

def metrics_block(df, scores, label):
    rows = []
    monsoon = df.site_timestamp.dt.month.isin([5, 6, 7, 8, 9, 10]).to_numpy()
    for (t, h), (m, prob) in scores.items():
        y = df.loc[m, f"y_ge{t}_{h}h"].to_numpy()
        pers = np.nan_to_num(df.loc[m, "fl_depth_now"].to_numpy()) >= t
        pred = (prob >= THRESHOLDS[f"ge{t}_{h}h"]) | pers        # hybrid
        row = {"target": f"ge{t}_{h}h", "scope": label,
               "positives": int(y.sum()),
               "pr_auc": average_precision_score(y, prob) if y.sum() else None}
        row["f2"], row["precision"], row["recall"] = f2_at(y, pred)
        pf2, _, _ = f2_at(y, pers)
        row["persistence_f2"] = pf2
        # monsoon subset
        mm = monsoon[m]
        ym, pm = y[mm], pred[mm]
        if ym.sum():
            row["monsoon_f2"], _, row["monsoon_recall"] = f2_at(ym, pm)
            row["monsoon_positives"] = int(ym.sum())
        rows.append(row)
    return pd.DataFrame(rows).round(4)

final = metrics_block(test, scores, "test-2025")
final

,target,scope,positives,pr_auc,f2,precision,recall,persistence_f2,monsoon_f2,monsoon_recall,monsoon_positives
0,ge5_1h,test-2025,2683,0.5837,0.5212,0.3301,0.6094,0.5789,0.4925,0.6158,1666
1,ge15_1h,test-2025,843,0.5600,0.5648,0.6435,0.5480,0.5527,0.5185,0.5078,447
2,ge30_1h,test-2025,119,0.0411,0.2242,0.0612,0.6723,0.5064,0.1053,0.6053,38
3,ge5_3h,test-2025,5330,0.2957,0.3134,0.3567,0.3041,0.3126,0.2967,0.2964,3394
4,ge15_3h,test-2025,1732,0.2711,0.3090,0.6038,0.2754,0.2887,0.2770,0.2482,959
5,ge30_3h,test-2025,264,0.0099,0.1113,0.0306,0.3258,0.2449,0.0443,0.2447,94
6,ge5_6h,test-2025,9221,0.1810,0.1940,0.3428,0.1750,0.1870,0.1853,0.1711,5886
7,ge15_6h,test-2025,3051,0.1602,0.1825,0.6758,0.1544,0.1698,0.1600,0.1355,1705
8,ge30_6h,test-2025,482,0.0728,0.1324,0.2995,0.1162,0.1378,0.0766,0.0730,178


## Depth quantiles on test (pinball, vs persistence)

In [6]:
def pinball(y, p, q):
    d = y - p
    return float(np.mean(np.maximum(q * d, (q - 1) * d)))

qrows = []
for h in HORIZONS:
    m = ((test[f"y_valid_{h}h"] >= VALID_MIN)
         & test[f"y_maxdepth_{h}h"].notna()).to_numpy()
    y = test.loc[m, f"y_maxdepth_{h}h"].to_numpy()
    pers = np.nan_to_num(test.loc[m, "fl_depth_now"].to_numpy())
    X = test.loc[m, FEATURES]
    for q in QUANTILES:
        qq = f"{int(q * 100):02d}"
        b = lgb.Booster(model_file=str(ARTIFACTS / f"reg_q{qq}_{h}h.txt"))
        pred = b.predict(X)
        qrows.append({"quantile": q, "horizon_h": h,
                      "pinball": pinball(y, pred, q),
                      "pinball_persistence": pinball(y, pers, q),
                      "coverage": float((y <= pred).mean())})
quant = pd.DataFrame(qrows).round(5)
quant

,quantile,horizon_h,pinball,pinball_persistence,coverage
0,0.05,1,0.00072,0.00091,0.99745
1,0.25,1,0.00226,0.00229,0.99789
2,0.50,1,0.00409,0.00401,0.99831
3,0.75,1,0.00561,0.00574,0.99881
4,0.95,1,0.00643,0.00712,0.99914
5,0.05,3,0.00139,0.00158,0.99506
6,0.25,3,0.00575,0.00573,0.99612
7,0.50,3,0.01098,0.01092,0.99694
8,0.75,3,0.01592,0.01610,0.99689
9,0.95,3,0.01955,0.02025,0.99773


## Per-station breakdown + save the final report

In [7]:
station_rows = []
codes_all = test["station_code"].astype(str)
for (t, h), (m, prob) in scores.items():
    y = test.loc[m, f"y_ge{t}_{h}h"].to_numpy()
    pers = np.nan_to_num(test.loc[m, "fl_depth_now"].to_numpy()) >= t
    pred = (prob >= THRESHOLDS[f"ge{t}_{h}h"]) | pers
    g = (pd.DataFrame({"station": codes_all[m].to_numpy(),
                       "y": y, "pred": pred})
         .groupby("station")
         .apply(lambda s: pd.Series({
             "pos": int(s.y.sum()),
             "tp": int((s.pred & (s.y == 1)).sum()),
             "fp": int((s.pred & (s.y == 0)).sum()),
             "fn": int((~s.pred & (s.y == 1)).sum())}),
                include_groups=False)
         .reset_index())
    g.insert(0, "target", f"ge{t}_{h}h")
    station_rows.append(g)
stations = pd.concat(station_rows, ignore_index=True)
stations.to_csv(ARTIFACTS / "final_station_breakdown.csv", index=False)

report = {
    "winner": WINNER,
    "test_split": "2025 (chronological holdout, thresholds frozen on val-2024)",
    "classification": final.to_dict(orient="records"),
    "depth_quantiles": quant.to_dict(orient="records"),
    "notes": [
        "hybrid rule applied: alert = model >= val-threshold OR depth >= tier now",
        "monsoon scope = May-Oct 2025",
        "run-once policy: these numbers are final; any later model change "
        "invalidates this as a held-out estimate",
    ],
}
(ARTIFACTS / "final_report.json").write_text(json.dumps(report, indent=2))
print("saved:", ARTIFACTS / "final_report.json")
print("saved:", ARTIFACTS / "final_station_breakdown.csv")

saved: ../models/artifacts/final_report.json
saved: ../models/artifacts/final_station_breakdown.csv


## (Only if WINNER = "gru") — GRU test inference

Self-contained re-definition matching `train_gru.ipynb`. Run this, then set `scores = score_gru(test)` and re-run the metric cells above.

In [8]:
def score_gru(df):
    import torch
    from torch import nn
    ckpt = torch.load(ARTIFACTS / "gru_model.pt", map_location="cpu")
    cfg = ckpt["config"]
    SEQ_LEN, SEQ_FEATURES = cfg["SEQ_LEN"], cfg["SEQ_FEATURES"]

    class DualHeadGRU(nn.Module):
        def __init__(self, n_feat, n_stations):
            super().__init__()
            self.emb = nn.Embedding(n_stations, cfg["EMB"])
            self.gru = nn.GRU(n_feat + cfg["EMB"], cfg["HIDDEN"],
                              batch_first=True)
            self.norm = nn.LayerNorm(cfg["HIDDEN"])
            self.head_cls = nn.Sequential(
                nn.Linear(cfg["HIDDEN"], cfg["HIDDEN"]), nn.ReLU(),
                nn.Linear(cfg["HIDDEN"], 9))
            self.head_reg = nn.Sequential(
                nn.Linear(cfg["HIDDEN"], cfg["HIDDEN"]), nn.ReLU(),
                nn.Linear(cfg["HIDDEN"], 15))

        def forward(self, seq, sid):
            e = self.emb(sid)[:, None, :].expand(-1, seq.shape[1], -1)
            h, _ = self.gru(torch.cat([seq, e], dim=2))
            return self.head_cls(self.norm(h[:, -1]))

    d = df.sort_values(["station_code", "site_timestamp"])
    X = d[SEQ_FEATURES].to_numpy(np.float32)
    miss = np.isnan(X[:, :2]).astype(np.float32)
    X = np.concatenate([np.nan_to_num(X), miss], axis=1)
    ts = d["site_timestamp"].astype("int64").to_numpy() // 10**9
    codes = d["station_code"].astype(str).to_numpy()
    sidx = ckpt["station_index"]
    sid = np.array([sidx.get(c, 0) for c in codes], dtype=np.int64)
    ok = np.zeros(len(d), dtype=bool)
    ok[SEQ_LEN - 1:] = ((codes[SEQ_LEN - 1:] == codes[:-(SEQ_LEN - 1)])
                        & (ts[SEQ_LEN - 1:] - ts[:-(SEQ_LEN - 1)]
                           == (SEQ_LEN - 1) * 900))
    anchors = np.where(ok)[0]

    net = DualHeadGRU(X.shape[1], len(sidx) + 1)
    net.load_state_dict(ckpt["state_dict"])
    net.eval()
    probs = np.empty((len(anchors), 9), np.float32)
    with torch.no_grad():
        for s in range(0, len(anchors), 8192):
            j = anchors[s: s + 8192]
            seq = np.stack([X[k - SEQ_LEN + 1: k + 1] for k in j])
            logits = net(torch.from_numpy(seq), torch.from_numpy(sid[j]))
            probs[s: s + 8192] = torch.sigmoid(logits).numpy()

    # map back to (tier, horizon) masks aligned with the sorted frame
    out, c = {}, 0
    pos_in_orig = d.index.to_numpy()[anchors]
    for h in cfg["HORIZONS"]:
        vm = (df[f"y_valid_{h}h"] >= VALID_MIN).to_numpy()
        for t in cfg["TIERS"]:
            m = np.zeros(len(df), dtype=bool)
            m[pos_in_orig] = True
            m &= vm
            sel = vm[pos_in_orig]
            out[(t, h)] = (m, probs[sel, c])
            c += 1
    return out

## Done

`final_report.json` + `final_station_breakdown.csv` are the report numbers. Headline for the write-up: hybrid F2 vs persistence F2 per horizon, monsoon subset, and the q95 pinball win — plus the honest caveats (universal thresholds, citywide water/flow features, no live weather).